# Implementación de LeNet con PyTorch sobre MNIST

## Librerias

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## ¿Qué es LeNet?

**LeNet** es una de las arquitecturas clásicas de redes convolucionales.  
La idea principal es procesar imágenes en varias etapas:

1. **Convolución**: extrae patrones locales como bordes, curvas y trazos.
2. **Función de activación**: introduce no linealidad.
3. **Pooling**: reduce el tamaño espacial y conserva la información más importante.
4. **Capas totalmente conectadas**: usan las características extraídas para clasificar la imagen.

Aqui la red recibe una imagen de un dígito y produce una salida de 10 clases, una por cada dígito del 0 al 9.

Carga y transformación de MNIST

Usaremos `torchvision.datasets.MNIST` para descargar el conjunto de entrenamiento y prueba.  
Aplicaremos la transformación `ToTensor()` para convertir cada imagen a tensor y escalar sus valores al rango adecuado para trabajar en PyTorch. MNIST puede descargarse directamente indicando `download=True`. :contentReference[oaicite:6]{index=6}

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Tamaño train:", len(train_dataset))
print("Tamaño test :", len(test_dataset))

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Etiqueta: {labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## ¿Por qué usar una CNN para MNIST?

MNIST es un problema de clasificación de imágenes.  
Una red neuronal convolucional es adecuada porque:

- aprovecha la estructura espacial de la imagen,
- detecta patrones locales,
- usa menos parámetros que una red totalmente conectada sobre toda la imagen,
- y suele generalizar mejor en tareas visuales simples.

Las convoluciones permiten que un mismo filtro se aplique a distintas regiones de la imagen, lo que ayuda a reconocer un patrón aunque aparezca en posiciones ligeramente distintas.  
PyTorch define esta operación mediante `Conv2d`. :contentReference[oaicite:2]{index=2}

## Definición de la arquitectura LeNet

La red se construirá con `nn.Module`.  
PyTorch usa `nn.Module` como clase base para definir modelos y submódulos de red neuronal. :contentReference[oaicite:7]{index=7}

Usaremos dos capas convolucionales (`nn.Conv2d`), operaciones de pooling y tres capas lineales finales.

## Arquitectura LeNet adaptada a MNIST

MNIST contiene imágenes en escala de grises, por lo que la entrada tiene **1 canal**. :contentReference[oaicite:3]{index=3}

Una versión clásica y práctica de LeNet para MNIST puede ser:

- Entrada: `1 x 28 x 28`
- Convolución 1: `1 -> 6` filtros de tamaño `5x5`
- Activación ReLU
- MaxPooling `2x2`
- Convolución 2: `6 -> 16` filtros de tamaño `5x5`
- Activación ReLU
- MaxPooling `2x2`
- Aplanado
- Capa lineal: `16*4*4 -> 120`
- ReLU
- Capa lineal: `120 -> 84`
- ReLU
- Capa lineal: `84 -> 10`

**Tamaños de salida**

Si la entrada es `28x28`:
- Después de `Conv1` con kernel `5`:  
  `28 - 5 + 1 = 24`  
  salida: `6 x 24 x 24`

- Después de `MaxPool(2,2)`:  
  `6 x 12 x 12`

- Después de `Conv2` con kernel `5`:  
  `12 - 5 + 1 = 8`  
  salida: `16 x 8 x 8`

- Después de `MaxPool(2,2)`:  
  `16 x 4 x 4`

Entonces, al aplanar, se obtienen:

$$
16 \times 4 \times 4 = 256
$$

valores de entrada para la primera capa totalmente conectada.


In [2]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Parte convolucional
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Parte fully connected
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        # Activación
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (batch, 1, 28, 28)
        x = self.pool(self.relu(self.conv1(x)))   # -> (batch, 6, 12, 12)
        x = self.pool(self.relu(self.conv2(x)))   # -> (batch, 16, 4, 4)
        
        x = torch.flatten(x, start_dim=1)         # -> (batch, 256)
        x = self.relu(self.fc1(x))                # -> (batch, 120)
        x = self.relu(self.fc2(x))                # -> (batch, 84)
        x = self.fc3(x)                           # -> (batch, 10)
        
        return x

model = LeNet().to(device)
print(model)

NameError: name 'device' is not defined

## Función de pérdida

Como se trata de una clasificación multiclase, usaremos **entropía cruzada** con `nn.CrossEntropyLoss`.  
En PyTorch, esta función recibe los **logits** del modelo y las etiquetas de clase como enteros. :contentReference[oaicite:4]{index=4}

Intuitivamente:

- si el modelo asigna alta probabilidad a la clase correcta, la pérdida baja;
- si se equivoca con alta confianza, la pérdida sube.

## Optimización

Para entrenar la red, se actualizan sus pesos minimizando la función de pérdida por medio de retropropagación y un optimizador.  
PyTorch agrupa estos métodos en `torch.optim`; por ejemplo, `Adam` o `SGD`. :contentReference[oaicite:5]{index=5}

En cada iteración de entrenamiento se sigue este flujo:

1. pasar imágenes por la red (**forward**),
2. calcular la pérdida,
3. limpiar gradientes anteriores,
4. hacer **backpropagation**,
5. actualizar pesos.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Entrenamiento

En cada época:

1. la red recibe un lote de imágenes,
2. calcula sus predicciones,
3. se evalúa la pérdida con `CrossEntropyLoss`,
4. se hace retropropagación,
5. el optimizador actualiza los pesos.

Esto corresponde al flujo estándar de entrenamiento supervisado en PyTorch.  
La pérdida de entropía cruzada compara los logits del modelo con la clase verdadera, y los optimizadores de `torch.optim` actualizan los parámetros entrenables. :contentReference[oaicite:8]{index=8}

In [ ]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=5):
    history = {
        "train_loss": [],
        "train_acc": []
    }
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        history["train_loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        
        print(f"Época [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")
    
    return history

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5
)

## Evaluacion

In [3]:
def evaluate_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_loss = running_loss / total
    test_acc = correct / total
    
    return test_loss, test_acc

test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

NameError: name 'model' is not defined

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_loss"], marker="o")
plt.title("Pérdida de entrenamiento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_acc"], marker="o")
plt.title("Accuracy de entrenamiento")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

## Ejemplo de predictores 

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"Real: {labels[i].item()}\nPred: {preds[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()